# IDC409: 01 Data Collection (Finance + News)

Run the cells **top to bottom**. The only thing you need to enter is your NewsAPI key (Step 2).

**Source notes (keep these for your README):**
- Prices: Yahoo Finance via `yfinance` (NVDA, GC=F gold, CL=F crude oil as petrol proxy, USDINR=X)
- News: NewsAPI (API key required; free plan covers about the last 30 days)
- Sentiment: VADER (compound score from -1 to +1)
- Limitation: retail petrol prices are not on yfinance, so crude oil futures are used as a proxy

## Step 1: Install libraries

In [1]:
!pip install yfinance vaderSentiment -q

## Step 2: Enter your NewsAPI key
Get a free key at https://newsapi.org/register. When you run the cell, a box appears: paste the key and press Enter.
The key is **not stored in the notebook**, so it is safe to upload to GitHub. The notebook stops if no key is entered.

In [2]:
from getpass import getpass

NEWS_API_KEY = getpass("Paste your NewsAPI key: ").strip()
assert NEWS_API_KEY, "No key entered. Re-run this cell and paste your NewsAPI key."
print("Key received.")

Paste your NewsAPI key: ··········
Key received.


## Step 3: Settings (change these if you like)

In [3]:
import pandas as pd, numpy as np, sqlite3, requests
from datetime import date, timedelta

TICKERS = {"stock": "NVDA", "gold": "GC=F", "oil": "CL=F", "usdinr": "USDINR=X"}
START_DATE = "2021-01-01"

QUERIES = ["Nvidia", "NVDA stock", "gold price", "crude oil", "rupee dollar", "Federal Reserve"]
KEYWORDS = ["ai", "chip", "earnings", "export", "tariff", "china", "record", "fall", "surge", "inflation", "rate", "ban"]

DB_PATH = "/content/market.db"
CSV_PATH = "/content/clean_data.csv"


## Step 4: Download prices (yfinance)
One column per asset, one row per day.

In [4]:
import yfinance as yf

raw = yf.download(list(TICKERS.values()), start=START_DATE, auto_adjust=True, progress=False)["Close"]
inv = {v: k for k, v in TICKERS.items()}
prices = raw.rename(columns=inv)[list(TICKERS.keys())]
prices.index = pd.to_datetime(prices.index).tz_localize(None)
prices.index.name = "date"

# Different markets have different holidays: carry the last known price forward
prices = prices.ffill().dropna()
print(prices.shape)
prices.tail()

(1514, 4)


Ticker,stock,gold,oil,usdinr
date,,,,
2026-10-22,212.210007,4060.399902,99.470001,94.312302
2026-10-23,210.960007,4007.899902,103.269997,94.711304
2026-10-26,216.000000,4038.399902,101.760002,94.938004
2026-10-27,217.940002,4074.600098,99.900002,95.186996
2026-10-28,217.139999,4052.899902,102.830002,95.209999


## Step 5: Collect news
Headlines come from NewsAPI. Each headline is stored with its date, source, search query and where it came from.

In [5]:
def fetch_newsapi(query, api_key, days_back=29):
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "from": (date.today() - timedelta(days=days_back)).isoformat(),
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 100,
        "apiKey": api_key,
    }
    data = requests.get(url, params=params, timeout=30).json()
    if data.get("status") != "ok":
        print(f"  NewsAPI problem for '{query}': {data.get('message')}")
        return []
    return [{"date": a["publishedAt"][:10], "headline": a["title"],
             "source": a["source"]["name"], "query": query, "origin": "newsapi"}
            for a in data["articles"] if a.get("title")]


items = []
for q in QUERIES:
    got = fetch_newsapi(q, NEWS_API_KEY)
    print(f"NewsAPI '{q}': {len(got)} headlines")
    items += got

assert items, "No headlines returned. Check your API key and the messages above."

news_new = pd.DataFrame(items)
print("Total collected:", len(news_new))
news_new.head()

NewsAPI 'Nvidia': 100 headlines
NewsAPI 'NVDA stock': 100 headlines
NewsAPI 'gold price': 99 headlines
NewsAPI 'crude oil': 98 headlines
NewsAPI 'rupee dollar': 100 headlines
NewsAPI 'Federal Reserve': 100 headlines
Total collected: 597


,date,headline,source,query,origin
0,2026-10-27,Nvidia Drops $3 Billion into Meta to Power its...,iPhone in Canada,Nvidia,newsapi
1,2026-10-27,Apple’s Most Expensive iPhone Is Missing This ...,Gizmodo.com,Nvidia,newsapi
2,2026-10-27,"Tech Giants Helping Finance Their Own AI Boom,...",The Daily Caller,Nvidia,newsapi
3,2026-10-27,Oracle Rises 5% as Google TPUs Come to Oracle ...,Biztoc.com,Nvidia,newsapi
4,2026-10-27,CineUp AI 1.1,Majorgeeks.com,Nvidia,newsapi


## Step 6: Clean, score sentiment, count keywords
Duplicate headlines are removed so nothing is counted twice.

In [6]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

news_new = news_new.drop_duplicates("headline").reset_index(drop=True)

sia = SentimentIntensityAnalyzer()
news_new["sentiment"] = news_new["headline"].apply(lambda h: sia.polarity_scores(h)["compound"])

for k in KEYWORDS:
    pattern = rf"\b{k}\b" if len(k) <= 3 else rf"\b{k}"   # short words like "ai" must match whole words only
    news_new[f"kw_{k}"] = news_new["headline"].str.lower().str.contains(pattern, regex=True).astype(int)

news_new.head()

,date,headline,source,query,origin,sentiment,kw_ai,kw_chip,kw_earnings,kw_export,kw_tariff,kw_china,kw_record,kw_fall,kw_surge,kw_inflation,kw_rate,kw_ban
0,2026-10-27,Nvidia Drops $3 Billion into Meta to Power its...,iPhone in Canada,Nvidia,newsapi,0.0000,0,0,0,0,0,0,0,0,0,0,0,0
1,2026-10-27,Apple’s Most Expensive iPhone Is Missing This ...,Gizmodo.com,Nvidia,newsapi,-0.2960,0,0,0,0,0,0,0,0,0,0,0,0
2,2026-10-27,"Tech Giants Helping Finance Their Own AI Boom,...",The Daily Caller,Nvidia,newsapi,0.0258,1,0,0,0,0,0,0,0,0,0,0,0
3,2026-10-27,Oracle Rises 5% as Google TPUs Come to Oracle ...,Biztoc.com,Nvidia,newsapi,0.2732,0,0,0,0,0,0,0,0,0,0,0,0
4,2026-10-27,CineUp AI 1.1,Majorgeeks.com,Nvidia,newsapi,0.0000,1,0,0,0,0,0,0,0,0,0,0,0


## Step 7: Save to SQLite
News is **appended** to whatever is already in the database, so re-running on another day grows your history.

In [7]:
conn = sqlite3.connect(DB_PATH)

# Prices: replace each time (always the full history)
prices.reset_index().to_sql("prices", conn, if_exists="replace", index=False)

# News: combine with old rows (if any) and drop duplicates
try:
    old = pd.read_sql("SELECT * FROM news", conn)
    news_all = pd.concat([old, news_new], ignore_index=True)
except Exception:
    news_all = news_new.copy()

news_all = news_all.drop_duplicates(["headline", "date"]).reset_index(drop=True)
news_all.to_sql("news", conn, if_exists="replace", index=False)

print("Prices rows:", len(prices), "| News rows:", len(news_all))
pd.read_sql("SELECT date, stock, gold FROM prices ORDER BY date DESC LIMIT 5", conn)

Prices rows: 1514 | News rows: 541


,date,stock,gold
0,2026-10-28 00:00:00,217.139999,4052.899902
1,2026-10-27 00:00:00,217.940002,4074.600098
2,2026-10-26 00:00:00,216.000000,4038.399902
3,2026-10-23 00:00:00,210.960007,4007.899902
4,2026-10-22 00:00:00,212.210007,4060.399902


## Step 8: Daily features, merge and export
- News on weekends is moved to the **next trading day**, because that is when the market can react.
- Days with no news get zeros, and `news_covered` marks the dates where news data actually exists, so you can model only that period if you want.

In [8]:
news_all["date"] = pd.to_datetime(news_all["date"])
news_all["trade_date"] = news_all["date"].apply(
    lambda d: d if d.weekday() < 5 else d + pd.offsets.BDay(1))

daily_news = news_all.groupby("trade_date").agg(
    n_headlines=("headline", "count"),
    avg_sentiment=("sentiment", "mean"),
    **{f"kw_{k}": (f"kw_{k}", "sum") for k in KEYWORDS}
).reset_index().rename(columns={"trade_date": "date"})

returns = prices.pct_change().add_suffix("_ret")
df = prices.join(returns).reset_index()
df = df.merge(daily_news, on="date", how="left")

news_start = news_all["date"].min()
df["news_covered"] = (df["date"] >= news_start).astype(int)

fill_cols = [c for c in daily_news.columns if c != "date"]
df[fill_cols] = df[fill_cols].fillna(0)
df = df.dropna().reset_index(drop=True)

df.to_csv(CSV_PATH, index=False)
df.to_sql("daily_features", conn, if_exists="replace", index=False)

print("Final table:", df.shape, "| news starts:", news_start.date())
df.tail()

Final table: (1513, 24) | news starts: 2026-10-01


,date,stock,gold,oil,usdinr,stock_ret,gold_ret,oil_ret,usdinr_ret,n_headlines,...,kw_export,kw_tariff,kw_china,kw_record,kw_fall,kw_surge,kw_inflation,kw_rate,kw_ban,news_covered
1508,2026-10-22,212.210007,4060.399902,99.470001,94.312302,-0.043970,-0.021590,0.023881,-0.004266,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1509,2026-10-23,210.960007,4007.899902,103.269997,94.711304,-0.005890,-0.012930,0.038202,0.004231,40.0,...,0.0,0.0,0.0,1.0,2.0,2.0,0.0,0.0,0.0,1
1510,2026-10-26,216.000000,4038.399902,101.760002,94.938004,0.023891,0.007610,-0.014622,0.002394,85.0,...,0.0,0.0,1.0,3.0,6.0,1.0,1.0,9.0,0.0,1
1511,2026-10-27,217.940002,4074.600098,99.900002,95.186996,0.008981,0.008964,-0.018278,0.002623,342.0,...,8.0,0.0,3.0,2.0,10.0,6.0,8.0,17.0,0.0,1
1512,2026-10-28,217.139999,4052.899902,102.830002,95.209999,-0.003671,-0.005326,0.029329,0.000242,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


## Step 9: Download your files
Colab deletes local files when the session ends, so download these now and upload them to the `data/` folder on GitHub.

In [9]:
from google.colab import files
files.download(DB_PATH)
files.download(CSV_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>